# RoCOCO CLIP Retrieval Analysis

Results table: R@1, RSMS, DropRate across Baseline / Isotropic / Manifold smoothing
for all 5 annotation types (Base COCO, Danger, Same, Diff, Rand).

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'axes.titlesize': 13,
    'legend.fontsize': 9, 'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})

C = {
    'baseline':  '#636363',
    'isotropic': '#2166ac',
    'manifold':  '#f5c518',
}
LS = {'baseline': ':', 'isotropic': '--', 'manifold': '-'}

REPO_ROOT = Path(os.path.abspath('..'))
OUT_DIR   = REPO_ROOT / 'output' / 'rococo'

MODES     = ['baseline', 'isotropic', 'manifold']
ANN_STEMS = ['coco_karpathy_test', 'danger', 'same_concept', 'diff_concept', 'rand_voca']
ANN_LABELS = {'coco_karpathy_test': 'Base COCO', 'danger': 'Danger',
               'same_concept': 'Same', 'diff_concept': 'Diff', 'rand_voca': 'Rand'}

def load_json(p):
    return json.loads(Path(p).read_text()) if Path(p).exists() else None

def sigma_tag(s): return f"sigma_{s:.2f}".replace('.','_')

# Load all results — both flat (no sigma subdir) and sigma-sweep subdir
results = {}   # (mode, ann, sigma) → metrics dict

for mode in MODES:
    mode_dir = OUT_DIR / mode
    if not mode_dir.exists(): continue

    # sigma subdirs: output/rococo/{mode}/sigma_X_XX/{ann}_metrics.json
    for sigma_dir in sorted(mode_dir.glob("sigma_*")):
        sigma_val = float(sigma_dir.name.replace("sigma_","").replace("_","."))
        for ann in ANN_STEMS:
            m = load_json(sigma_dir / f"{ann}_metrics.json")
            if m:
                results[(mode, ann, sigma_val)] = m

    # flat (no sigma subdir): output/rococo/{mode}/{ann}_metrics.json  (single sigma)
    for ann in ANN_STEMS:
        m = load_json(mode_dir / f"{ann}_metrics.json")
        if m:
            sigma_val = m.get("smoothing", {}).get("sigma", 0.0)
            results[(mode, ann, sigma_val)] = m

sigmas_found = sorted(set(s for _,_,s in results.keys()))
print(f"Loaded {len(results)} results")
print(f"Modes:  {MODES}")
print(f"Sigmas: {sigmas_found}")

## 1. Main Results Table

In [ ]:
## Best-sigma results table
# For each (mode, ann): pick the sigma with highest R@1

rows = []
for ann in ANN_STEMS:
    row = {'Dataset': ANN_LABELS[ann]}
    for mode in MODES:
        best = max(
            [results[(mode,ann,s)] for s in sigmas_found if (mode,ann,s) in results],
            key=lambda m: m.get('R@1', 0), default={}
        )
        row[f'{mode.title()} R@1']    = best.get('R@1',  None)
        row[f'{mode.title()} RSMS']   = best.get('RSMS', None)
        row[f'{mode.title()} σ*']     = best.get('smoothing',{}).get('sigma', None)
    rows.append(row)

df = pd.DataFrame(rows).set_index('Dataset')
fmt = lambda v: f'{v:.1f}' if v is not None else '–'
display(df.style.format(fmt)
        .background_gradient(subset=[c for c in df.columns if 'R@1' in c], cmap='Blues', vmin=0, vmax=100)
        .background_gradient(subset=[c for c in df.columns if 'RSMS' in c], cmap='Reds_r', vmin=0, vmax=30)
        .set_caption('Best-σ results per method'))

In [ ]:
## Compute DropRate = (R@1_base - R@1_ann) / R@1_base
# Requires: coco_karpathy_test result for the same mode

drop_rows = []
for ann in [a for a in ANN_STEMS if a != 'coco_karpathy_test']:
    for mode in MODES:
        base = results.get((mode, 'coco_karpathy_test'), {})
        adv  = results.get((mode, ann), {})
        r1_base = base.get('R@1', None)
        r1_adv  = adv.get('R@1',  None)
        drop = round((r1_base - r1_adv) / r1_base * 100, 2) if (r1_base and r1_adv) else None
        drop_rows.append({
            'Dataset': ANN_LABELS[ann],
            'Mode':    mode.title(),
            'R@1 Base (COCO)': r1_base,
            'R@1 Ann':         r1_adv,
            'DropRate (%)':    drop,
            'RSMS (%)':        adv.get('RSMS', None),
        })

drop_df = pd.DataFrame(drop_rows)
display(drop_df.style
        .format({'R@1 Base (COCO)': '{:.1f}', 'R@1 Ann': '{:.1f}',
                 'DropRate (%)': '{:.1f}', 'RSMS (%)': '{:.1f}'})
        .background_gradient(subset=['DropRate (%)'], cmap='Reds', vmin=0, vmax=30)
        .set_caption('DropRate = (R@1_COCO − R@1_RoCOCO) / R@1_COCO  ×100%'))

## 2. Figure — R@1 Comparison

In [ ]:
x = np.arange(len(ANN_STEMS))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for k, mode in enumerate(MODES):
    vals = [results.get((mode, ann), {}).get('R@1', np.nan) for ann in ANN_STEMS]
    ax.bar(x + (k-1)*w, vals, w, label=mode.title(), color=C[mode], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([ANN_LABELS[a] for a in ANN_STEMS])
ax.set_ylabel('R@1 (%)')
ax.set_title('Image-to-Text Retrieval R@1 — Baseline vs Isotropic vs Manifold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylim(0, 105)
ax.legend()
plt.tight_layout()
plt.savefig('fig_rococo_r1.pdf', bbox_inches='tight')
plt.show()

## 3. Figure — RSMS and DropRate

In [ ]:
adv_stems = [a for a in ANN_STEMS if a != 'coco_karpathy_test']
x = np.arange(len(adv_stems))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ['RSMS', 'DropRate'],
                              ['RSMS (%) — Adversarial caption top-1 rate',
                               'Drop Rate (%) — Relative R@1 decrease']):
    for k, mode in enumerate(MODES):
        vals = [results.get((mode, ann), {}).get(metric, np.nan) for ann in adv_stems]
        ax.bar(x + (k-1)*w, vals, w, label=mode.title(), color=C[mode], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([ANN_LABELS[a] for a in adv_stems])
    ax.set_ylabel(f'{metric} (%)')
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.legend()

plt.tight_layout()
plt.savefig('fig_rococo_rsms_drop.pdf', bbox_inches='tight')
plt.show()

## 4. Sigma Sweep (if multiple sigmas run)

In [ ]:
## R@1 vs σ for each annotation set (ISO vs Manifold)
if sigmas_found:
    fig, axes = plt.subplots(1, len(ANN_STEMS), figsize=(5*len(ANN_STEMS), 4.5),
                             sharey=False, squeeze=False)
    for j, ann in enumerate(ANN_STEMS):
        ax = axes[0, j]
        for mode in ['isotropic', 'manifold']:
            sigs = sorted(s for m,a,s in results if m==mode and a==ann)
            r1s  = [results[(mode,ann,s)].get('R@1', np.nan) for s in sigs]
            ax.plot(sigs, r1s, 'o-', color=C[mode], ls=LS[mode], lw=2, label=mode.title())
        # Baseline (sigma-independent) — horizontal line at best sigma
        base_vals = [results[('baseline',ann,s)].get('R@1') for s in sigmas_found
                     if ('baseline',ann,s) in results]
        if base_vals:
            ax.axhline(max(v for v in base_vals if v), color=C['baseline'],
                       lw=1.5, ls=':', label='Baseline')
        ax.set_title(ANN_LABELS[ann], fontweight='bold')
        ax.set_xlabel('σ')
        ax.set_ylabel('R@1 (%)' if j==0 else '')
        ax.legend(fontsize=8)

    fig.suptitle('R@1 vs σ — Isotropic vs Manifold smoothing', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig_rococo_sigma_sweep.pdf', bbox_inches='tight')
    plt.show()

    ## RSMS vs σ (adversarial annotations only)
    adv_anns = [a for a in ANN_STEMS if a != 'coco_karpathy_test']
    fig2, axes2 = plt.subplots(1, len(adv_anns), figsize=(5*len(adv_anns), 4.5),
                               sharey=False, squeeze=False)
    for j, ann in enumerate(adv_anns):
        ax = axes2[0, j]
        for mode in ['isotropic', 'manifold']:
            sigs = sorted(s for m,a,s in results if m==mode and a==ann)
            rsms = [results[(mode,ann,s)].get('RSMS', np.nan) for s in sigs]
            ax.plot(sigs, rsms, 'o-', color=C[mode], ls=LS[mode], lw=2, label=mode.title())
        ax.set_title(ANN_LABELS[ann], fontweight='bold')
        ax.set_xlabel('σ')
        ax.set_ylabel('RSMS (%)' if j==0 else '')
        ax.legend(fontsize=8)
    fig2.suptitle('RSMS vs σ — lower is better (fewer adversarial captions retrieved)', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig_rococo_rsms_sigma.pdf', bbox_inches='tight')
    plt.show()
else:
    print('No sigma sweep results yet.')